# Complete Data Inventory - All Data Sources

**Purpose**: Systematically explore EVERY data file in `backend/data/` to document:
- File types and counts
- Schemas (layers, fields, data types)
- Sample data
- Data quality issues
- Visualizations

**Output**: This notebook generates the content for `docs/DATA_SOURCES.md`

**Workflow**:
1. Discover all files by type
2. Explore each file type systematically
3. Document findings
4. Update DATA_SOURCES.md with actual schemas

---

**⚠️ IMPORTANT**: Clear all cell output before committing!


## 1. Setup & Imports


In [1]:
import sys
sys.path.append("../")

import geopandas as gpd
import pandas as pd
import folium
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import src.paths as PATHS
import src.constants as CONST
import src.config as CONFIG
from src.data.data_collector import DataCollector

print(f"✅ Setup complete")
print(f"📁 Data directory: {PATHS.DATA_DIR}")
print(f"🗺️  CRS: RD New = EPSG:{CONST.EPSG_RD}, WGS84 = EPSG:{CONST.EPSG_WGS84}")


✅ Setup complete
📁 Data directory: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data
🗺️  CRS: RD New = EPSG:28992, WGS84 = EPSG:4326


## 2. Discovery Phase - Scan All Files


In [2]:
# Discover ALL files in data directory
all_files = [f for f in PATHS.DATA_DIR.rglob("*") if f.is_file()]

# Group by extension
files_by_type = {}
for f in all_files:
    ext = f.suffix.lower()
    if ext not in files_by_type:
        files_by_type[ext] = []
    files_by_type[ext].append(f)

# Sort for consistent display
files_by_type = dict(sorted(files_by_type.items()))

print(f"📊 Found {len(all_files)} total files\n")
print("Files by type:")
print("=" * 60)
for ext, files in files_by_type.items():
    ext_display = ext if ext else "(no extension)"
    print(f"{ext_display:20} {len(files):>4} files")
    
print("\n" + "=" * 60)
print("\n💡 Focus on geospatial formats: .gpkg, .geojson, .geojsonl, .shp")


📊 Found 5464 total files

Files by type:
(no extension)          5 files
.cpg                    1 files
.csv                    2 files
.dbf                    1 files
.geojson                1 files
.geojsonl               3 files
.gif                   29 files
.gpkg                  10 files
.png                 5403 files
.prj                    1 files
.shp                    1 files
.shx                    1 files
.xlsx                   1 files
.zip                    5 files


💡 Focus on geospatial formats: .gpkg, .geojson, .geojsonl, .shp


## 3. Exploration Functions

Helper functions to systematically explore each file type.


In [ ]:
def explore_geopackage(gpkg_path: Path, max_rows: int = 5):
    """
    Comprehensively explore a GeoPackage file.
    
    Args:
        gpkg_path: Path to .gpkg file
        max_rows: Number of sample rows to display per layer
    
    Returns:
        dict: {layer_name: GeoDataFrame}
    """
    print(f"\n{'='*80}")
    print(f"📦 FILE: {gpkg_path.relative_to(PATHS.DATA_DIR)}")
    print(f"{'='*80}")
    
    if not gpkg_path.exists():
        print(f"❌ File not found")
        return {}
    
    # Get file size
    size_mb = gpkg_path.stat().st_size / (1024 * 1024)
    print(f"💾 Size: {size_mb:.2f} MB")
    
    # List all layers
    try:
        layers_df = gpd.list_layers(gpkg_path)
        print(f"\n📋 Layers: {len(layers_df)}")
        for idx, row in layers_df.iterrows():
            geom_type = row.get('geometry_type', 'Unknown')
            print(f"   {idx+1}. {row['name']:<40} ({geom_type})")
    except Exception as e:
        print(f"❌ Error listing layers: {e}")
        return {}
    
    # Explore each layer
    layers_data = {}
    for idx, layer_info in layers_df.iterrows():
        layer_name = layer_info['name']
        print(f"\n{'-'*80}")
        print(f"📍 Layer: {layer_name}")
        print(f"{'-'*80}")
        
        try:
            gdf = gpd.read_file(gpkg_path, layer=layer_name)
            
            # Basic info
            print(f"Shape: {gdf.shape[0]:,} rows × {gdf.shape[1]} columns")
            print(f"CRS: {gdf.crs}")
            print(f"Geometry types: {dict(gdf.geometry.geom_type.value_counts())}")
            print(f"Bounds: {gdf.total_bounds}")
            
            # Column details
            print(f"\nColumns:")
            for col in gdf.columns:
                dtype = gdf[col].dtype
                non_null = gdf[col].notna().sum()
                pct_null = 100 * (1 - non_null / len(gdf))
                
                # Sample unique values for categorical-looking columns
                if dtype == 'object' and col != 'geometry':
                    n_unique = gdf[col].nunique()
                    if n_unique <= 10:
                        unique_vals = list(gdf[col].unique()[:10])
                        print(f"  • {col:<30} {str(dtype):<12} {pct_null:>5.1f}% null  [{n_unique} unique: {unique_vals}]")
                    else:
                        print(f"  • {col:<30} {str(dtype):<12} {pct_null:>5.1f}% null  [{n_unique} unique values]")
                else:
                    print(f"  • {col:<30} {str(dtype):<12} {pct_null:>5.1f}% null")
            
            # Sample data (drop geometry for readability)
            print(f"\nSample data (first {max_rows} rows):")
            display_df = gdf.drop(columns=['geometry']) if 'geometry' in gdf.columns else gdf
            print(display_df.head(max_rows).to_string())
            
            layers_data[layer_name] = gdf
            
        except Exception as e:
            print(f"❌ Error reading layer: {e}")
    
    print(f"\n{'='*80}\n")
    return layers_data


In [14]:
def explore_geojson(geojson_path: Path, max_rows: int = 5):
    """Explore a GeoJSON file."""
    print(f"\n{'='*80}")
    print(f"🗺️  FILE: {geojson_path.relative_to(PATHS.DATA_DIR)}")
    print(f"{'='*80}")
    
    if not geojson_path.exists():
        print(f"❌ File not found")
        return None
    
    size_mb = geojson_path.stat().st_size / (1024 * 1024)
    print(f"💾 Size: {size_mb:.2f} MB")
    
    try:
        gdf = gpd.read_file(geojson_path)
        
        print(f"\nShape: {gdf.shape[0]:,} rows × {gdf.shape[1]} columns")
        print(f"CRS: {gdf.crs}")
        print(f"Geometry types: {dict(gdf.geometry.geom_type.value_counts())}")
        print(f"Bounds: {gdf.total_bounds}")
        
        print(f"\nColumns:")
        for col in gdf.columns:
            dtype = gdf[col].dtype
            non_null = gdf[col].notna().sum()
            pct_null = 100 * (1 - non_null / len(gdf))
            print(f"  • {col:<30} {str(dtype):<12} {pct_null:>5.1f}% null")
        
        print(f"\nSample data (first {max_rows} rows):")
        display_df = gdf.drop(columns=['geometry']) if 'geometry' in gdf.columns else gdf
        print(display_df.head(max_rows).to_string())
        
        print(f"\n{'='*80}\n")
        return gdf
        
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        print(f"{'='*80}\n")
        return None


def explore_geojsonl(geojsonl_path: Path, max_lines: int = 10):
    """Explore a GeoJSON Lines file (one feature per line)."""
    print(f"\n{'='*80}")
    print(f"📄 FILE: {geojsonl_path.relative_to(PATHS.DATA_DIR)}")
    print(f"{'='*80}")
    
    if not geojsonl_path.exists():
        print(f"❌ File not found")
        return None
    
    size_mb = geojsonl_path.stat().st_size / (1024 * 1024)
    print(f"💾 Size: {size_mb:.2f} MB")
    
    try:
        # Count lines
        with open(geojsonl_path, 'r') as f:
            num_lines = sum(1 for _ in f)
        
        print(f"\n📊 Total features: {num_lines:,}")
        
        # Parse sample features
        print(f"\nSample features (first {max_lines}):")
        features = []
        with open(geojsonl_path, 'r') as f:
            for i, line in enumerate(f):
                if i >= max_lines:
                    break
                try:
                    feature = json.loads(line)
                    features.append(feature)
                    print(f"\n  Feature {i+1}:")
                    print(f"    Type: {feature.get('type')}")
                    if 'geometry' in feature:
                        print(f"    Geometry: {feature['geometry'].get('type')}")
                    if 'properties' in feature:
                        print(f"    Properties: {list(feature['properties'].keys())}")
                except json.JSONDecodeError as e:
                    print(f"    ❌ Line {i+1}: JSON decode error")
        
        print(f"\n{'='*80}\n")
        return features
        
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        print(f"{'='*80}\n")
        return None


def explore_shapefile(shp_path: Path, max_rows: int = 5):
    """Explore a Shapefile."""
    print(f"\n{'='*80}")
    print(f"📐 FILE: {shp_path.relative_to(PATHS.DATA_DIR)}")
    print(f"{'='*80}")
    
    # Check for companion files
    companions = {
        '.shx': shp_path.with_suffix('.shx').exists(),
        '.dbf': shp_path.with_suffix('.dbf').exists(),
        '.prj': shp_path.with_suffix('.prj').exists(),
        '.cpg': shp_path.with_suffix('.cpg').exists(),
    }
    
    print(f"Companion files:")
    for ext, exists in companions.items():
        status = "✅" if exists else "❌"
        print(f"  {status} {ext}")
    
    if not companions['.shx'] or not companions['.dbf']:
        print(f"\n⚠️  Missing required companion files!")
        print(f"{'='*80}\n")
        return None
    
    try:
        gdf = gpd.read_file(shp_path)
        
        size_mb = shp_path.stat().st_size / (1024 * 1024)
        print(f"\n💾 .shp Size: {size_mb:.2f} MB")
        print(f"Shape: {gdf.shape[0]:,} rows × {gdf.shape[1]} columns")
        print(f"CRS: {gdf.crs}")
        print(f"Geometry types: {dict(gdf.geometry.geom_type.value_counts())}")
        print(f"Bounds: {gdf.total_bounds}")
        
        print(f"\nColumns:")
        for col in gdf.columns:
            dtype = gdf[col].dtype
            non_null = gdf[col].notna().sum()
            pct_null = 100 * (1 - non_null / len(gdf))
            print(f"  • {col:<30} {str(dtype):<12} {pct_null:>5.1f}% null")
        
        print(f"\nSample data (first {max_rows} rows):")
        display_df = gdf.drop(columns=['geometry']) if 'geometry' in gdf.columns else gdf
        print(display_df.head(max_rows).to_string())
        
        print(f"\n{'='*80}\n")
        return gdf
        
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        print(f"{'='*80}\n")
        return None


def explore_csv(csv_path: Path, max_rows: int = 10):
    """Explore a CSV file."""
    print(f"\n{'='*80}")
    print(f"📊 FILE: {csv_path.relative_to(PATHS.DATA_DIR)}")
    print(f"{'='*80}")
    
    size_mb = csv_path.stat().st_size / (1024 * 1024)
    print(f"💾 Size: {size_mb:.2f} MB")
    
    try:
        df = pd.read_csv(csv_path)
        
        print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        
        print(f"\nColumns:")
        for col in df.columns:
            dtype = df[col].dtype
            non_null = df[col].notna().sum()
            pct_null = 100 * (1 - non_null / len(df))
            print(f"  • {col:<30} {str(dtype):<12} {pct_null:>5.1f}% null")
        
        # Check for potential spatial columns
        spatial_cols = [c for c in df.columns if any(x in c.lower() for x in ['lat', 'lon', 'x', 'y', 'coord', 'geom'])]
        if spatial_cols:
            print(f"\n💡 Potential spatial columns: {spatial_cols}")
        
        print(f"\nSample data (first {max_rows} rows):")
        print(df.head(max_rows).to_string())
        
        print(f"\nBasic statistics:")
        print(df.describe().to_string())
        
        print(f"\n{'='*80}\n")
        return df
        
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        print(f"{'='*80}\n")
        return None


## 4. Explore All GeoPackage Files

Run exploration for each .gpkg file found.


In [15]:
# Get all .gpkg files
gpkg_files = files_by_type.get('.gpkg', [])
print(f"🔍 Exploring {len(gpkg_files)} GeoPackage files...\n")
for gpkg in gpkg_files:
    print(gpkg.name)

# Store results
all_gpkg_data = {}

# Explore each one
for gpkg_path in sorted(gpkg_files):
    layers_data = explore_geopackage(gpkg_path)
    if layers_data:
        rel_path = str(gpkg_path.relative_to(PATHS.DATA_DIR))
        all_gpkg_data[rel_path] = layers_data

print(f"\n✅ Explored {len(all_gpkg_data)} GeoPackage files successfully")


🔍 Exploring 10 GeoPackage files...

all_results_20250121_v2.gpkg
Levering_erosie_data.gpkg
Vlak_Vrijeruimte_ln.gpkg
erosion_border_20250129.gpkg
draft_oever_degradation_data.gpkg
phase1_2025-08-14_v1.gpkg
luke_inputs_v3.gpkg
erosion_locations_sam.gpkg
sam_processed.gpkg
phase1_2025-07-15_v3.gpkg

📦 FILE: Levering_erosie_data.gpkg
💾 Size: 9.21 MB

📋 Layers: 7
   1. Rivier                                   (Polygon)
   2. Kribben_BKN                              (Polygon)
   3. Vlak_voor_vrijeruimte_natuurvriendelijke_oever (Polygon)
   4. Vlak_voor_vrijeruimte_nevengeulen        (Polygon)
   5. Uiterwaardegrenzen                       (Polygon)
   6. Kilometrering                            (Point)
   7. Centreline_River                         (LineString)

--------------------------------------------------------------------------------
📍 Layer: Rivier
--------------------------------------------------------------------------------
Shape: 11 rows × 49 columns
CRS: EPSG:28992
Geometry t

## 5. Explore GeoJSON Files

Single-layer geospatial files.


In [ ]:
# Get all .geojson files
geojson_files = files_by_type.get('.geojson', [])
print(f"🔍 Exploring {len(geojson_files)} GeoJSON files...\n")

all_geojson_data = {}
for geojson_path in sorted(geojson_files):
    gdf = explore_geojson(geojson_path)
    if gdf is not None:
        rel_path = str(geojson_path.relative_to(PATHS.DATA_DIR))
        all_geojson_data[rel_path] = gdf

print(f"\n✅ Explored {len(all_geojson_data)} GeoJSON files successfully")


🔍 Exploring 1 GeoJSON files...


🗺️  FILE: handdrawn_fake_erosion_border.geojson
💾 Size: 0.00 MB

Shape: 1 rows × 1 columns
CRS: EPSG:4326
Geometry types: {'LineString': np.int64(1)}
Bounds: [ 5.039206 51.814372  5.068774 51.821668]

Columns:
  • geometry                       geometry       0.0% null

Sample data (first 5 rows):
Empty DataFrame
Columns: []
Index: [0]



✅ Explored 1 GeoJSON files successfully


## 6. Explore GeoJSON Lines Files (.geojsonl)

SAM annotation format - one feature per line.


In [7]:
# Get all .geojsonl files
geojsonl_files = files_by_type.get('.geojsonl', [])
print(f"🔍 Exploring {len(geojsonl_files)} GeoJSON Lines files...\n")

all_geojsonl_data = {}
for geojsonl_path in sorted(geojsonl_files):
    features = explore_geojsonl(geojsonl_path)
    if features is not None:
        rel_path = str(geojsonl_path.relative_to(PATHS.DATA_DIR))
        all_geojsonl_data[rel_path] = features

print(f"\n✅ Explored {len(all_geojsonl_data)} GeoJSON Lines files successfully")


🔍 Exploring 3 GeoJSON Lines files...


📄 FILE: sam/Brakel.geojsonl
💾 Size: 35.42 MB

📊 Total features: 937

Sample features (first 10):

  Feature 1:
    Type: Feature
    Geometry: Polygon
    Properties: ['observation_id', 'patch_id', 'observation_date', 'water_height_m', 'rotation_angle_deg', 'source_image_filename', 'mask_image_filename', 'soil_image_filename']

  Feature 2:
    Type: Feature
    Geometry: Polygon
    Properties: ['observation_id', 'patch_id', 'observation_date', 'water_height_m', 'rotation_angle_deg', 'source_image_filename', 'mask_image_filename', 'soil_image_filename']

  Feature 3:
    Type: Feature
    Geometry: Polygon
    Properties: ['observation_id', 'patch_id', 'observation_date', 'water_height_m', 'rotation_angle_deg', 'source_image_filename', 'mask_image_filename', 'soil_image_filename']

  Feature 4:
    Type: Feature
    Geometry: Polygon
    Properties: ['observation_id', 'patch_id', 'observation_date', 'water_height_m', 'rotation_angle_deg', 'source

## 7. Explore Shapefiles (.shp)

Legacy format with multiple companion files.


In [ ]:
# Get all .shp files
shp_files = files_by_type.get('.shp', [])
print(f"🔍 Exploring {len(shp_files)} Shapefile(s)...\n")

all_shp_data = {}
for shp_path in sorted(shp_files):
    gdf = explore_shapefile(shp_path)
    if gdf is not None:
        rel_path = str(shp_path.relative_to(PATHS.DATA_DIR))
        all_shp_data[rel_path] = gdf

print(f"\n✅ Explored {len(all_shp_data)} Shapefile(s) successfully")


🔍 Exploring 1 Shapefile(s)...


📐 FILE: VO155184_Scope_Pilot_Bankerosion_20241129/VO155184_Scope_Pilot_Bankerosion_20241129.shp
Companion files:
  ✅ .shx
  ✅ .dbf
  ✅ .prj
  ✅ .cpg

💾 .shp Size: 0.04 MB
Shape: 4 rows × 8 columns
CRS: EPSG:3857
Geometry types: {'Polygon': np.int64(4)}
Bounds: [ 557041.92382524 6763152.36686913  577428.48242568 6768856.74158206]

Columns:
  • naam                           object         0.0% null
  • code                           object         0.0% null
  • altschrijf                     object        75.0% null
  • altschrij0                     object       100.0% null
  • altnaam_1                      object        25.0% null
  • altnaam_2                      object       100.0% null
  • bewerkings                     object         0.0% null
  • geometry                       geometry       0.0% null

Sample data (first 5 rows):
                        naam      code               altschrijf altschrij0            altnaam_1 altnaam_2            b

## 8. Explore CSV Files

Tabular data (may contain spatial info).


In [ ]:
# Get all .csv files
csv_files = files_by_type.get('.csv', [])
print(f"🔍 Exploring {len(csv_files)} CSV file(s)...\n")

all_csv_data = {}
for csv_path in sorted(csv_files):
    df = explore_csv(csv_path)
    if df is not None:
        rel_path = str(csv_path.relative_to(PATHS.DATA_DIR))
        all_csv_data[rel_path] = df

print(f"\n✅ Explored {len(all_csv_data)} CSV file(s) successfully")


🔍 Exploring 2 CSV file(s)...


📊 FILE: water_data/20250516_016.csv
💾 Size: 0.13 MB

Shape: 366 rows × 1 columns

Columns:
  • MONSTER_IDENTIFICATIE;MEETPUNT_IDENTIFICATIE;LOCATIE_CODE;TYPERING_OMSCHRIJVING;TYPERING_CODE;GROOTHEID_OMSCHRIJVING;GROOTHEID_ CODE;PARAMETER_OMSCHRIJVING;PARAMETER_ CODE;CAS_NR;EENHEID_CODE;HOEDANIGHEID_OMSCHRIJVING;HOEDANIGHEID_CODE;COMPARTIMENT_OMSCHRIJVING;COMPARTIMENT_CODE;WAARDEBEWERKINGSMETHODE_OMSCHRIJVING;WAARDEBEWERKINGSMETHODE_CODE;WAARDEBEPALINGSMETHODE_OMSCHRIJVING;WAARDEBEPALINGSMETHODE_CODE;BEMONSTERINGSSOORT_OMSCHRIJVING;BEMONSTERINGSSOORT_CODE;WAARNEMINGDATUM;WAARNEMINGTIJD (MET/CET);LIMIETSYMBOOL;NUMERIEKEWAARDE;ALFANUMERIEKEWAARDE;KWALITEITSOORDEEL_CODE;REFERENTIE;NOTITIE_CODE;NOTITIE_OMSCHRIJVING;STATUSWAARDE;OPDRACHTGEVENDE_INSTANTIE;MEETAPPARAAT_OMSCHRIJVING;MEETAPPARAAT_CODE;BEMONSTERINGSAPPARAAT_OMSCHRIJVING;BEMONSTERINGSAPPARAAT_CODE;PLAATSBEPALINGSAPPARAAT_OMSCHRIJVING;PLAATSBEPALINGSAPPARAAT_CODE;BEMONSTERINGSHOOGTE;REFERENTIEVLAK;EPS

## 9. Other Files

List remaining file types for reference.


In [ ]:
# List other file types found
explored_types = {'.gpkg', '.geojson', '.geojsonl', '.shp', '.csv'}
other_types = {ext: files for ext, files in files_by_type.items() if ext not in explored_types}

if other_types:
    print("📂 Other file types found:\n")
    for ext, files in sorted(other_types.items()):
        ext_display = ext if ext else "(no extension)"
        print(f"\n{ext_display} ({len(files)} files):")
        for f in sorted(files)[:10]:  # Show first 10
            print(f"  - {f.relative_to(PATHS.DATA_DIR)}")
        if len(files) > 10:
            print(f"  ... and {len(files) - 10} more")
else:
    print("✅ All relevant file types explored!")


📂 Other file types found:


(no extension) (5 files):
  - .DS_Store
  - luke_for_feedback/.DS_Store
  - luke_for_feedback/plots/.DS_Store
  - sam/.DS_Store
  - water_data/.DS_Store

.cpg (1 files):
  - VO155184_Scope_Pilot_Bankerosion_20241129/VO155184_Scope_Pilot_Bankerosion_20241129.cpg

.dbf (1 files):
  - VO155184_Scope_Pilot_Bankerosion_20241129/VO155184_Scope_Pilot_Bankerosion_20241129.dbf

.gif (29 files):
  - luke_for_feedback/plots/ijssel_l_9493_9494/elevation_profiles.gif
  - luke_for_feedback/plots/ijssel_r_9490_9491/elevation_profiles.gif
  - luke_for_feedback/plots/ijssel_r_9508_9509/elevation_profiles.gif
  - luke_for_feedback/plots/maas_l_2198_2199/elevation_profiles.gif
  - luke_for_feedback/plots/maas_l_2199_2200/elevation_profiles.gif
  - luke_for_feedback/plots/maas_l_2238_2239/elevation_profiles.gif
  - luke_for_feedback/plots/maas_r_2201_2202/elevation_profiles.gif
  - luke_for_feedback/plots/maas_r_2210_2211/elevation_profiles.gif
  - luke_for_feedback/plots/maas_

## 10. Summary & Next Steps


In [ ]:
print("\n" + "="*80)
print("📊 DATA INVENTORY SUMMARY")
print("="*80)

print(f"\n✅ Explored Files:")
print(f"   • GeoPackage files (.gpkg):     {len(all_gpkg_data)}")
print(f"   • GeoJSON files (.geojson):     {len(all_geojson_data)}")
print(f"   • GeoJSON Lines (.geojsonl):    {len(all_geojsonl_data)}")
print(f"   • Shapefiles (.shp):            {len(all_shp_data)}")
print(f"   • CSV files (.csv):             {len(all_csv_data)}")

print(f"\n📋 Total Layers in GeoPackages: {sum(len(layers) for layers in all_gpkg_data.values())}")

print(f"\n💡 Next Steps:")
print("   1. Review the output above for each file")
print("   2. Note any data quality issues (missing values, wrong CRS, etc.)")
print("   3. Update docs/DATA_SOURCES.md with:")
print("      - Actual layer names and schemas")
print("      - Field descriptions")
print("      - Data quality notes")
print("   4. Create visualizations if needed (Folium maps)")
print("   5. Document any files that are duplicates or can be archived")

print(f"\n⚠️  REMEMBER: Clear all cell output before committing!")
print("="*80)



📊 DATA INVENTORY SUMMARY

✅ Explored Files:
   • GeoPackage files (.gpkg):     10
   • GeoJSON files (.geojson):     1
   • GeoJSON Lines (.geojsonl):    3
   • Shapefiles (.shp):            1
   • CSV files (.csv):             2

📋 Total Layers in GeoPackages: 34

💡 Next Steps:
   1. Review the output above for each file
   2. Note any data quality issues (missing values, wrong CRS, etc.)
   3. Update docs/DATA_SOURCES.md with:
      - Actual layer names and schemas
      - Field descriptions
      - Data quality notes
   4. Create visualizations if needed (Folium maps)
   5. Document any files that are duplicates or can be archived

⚠️  REMEMBER: Clear all cell output before committing!
